In [1]:
!pip install faster-whisper sounddevice scipy pyttsx3 transformers torch



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sounddevice as sd
from scipy.io.wavfile import write
from faster_whisper import WhisperModel
import pyttsx3
from transformers import pipeline


In [13]:
# Speech to Text
whisper_model = WhisperModel(
    "base",
    device="cpu",
    compute_type="int8"
)

# Question Answering Model
chatbot = pipeline(
    "text-generation",
    model="microsoft/DialoGPT-small"
)

# Text to Speech
speaker = pyttsx3.init()

config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

C:\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\praka\.cache\huggingface\hub\models--microsoft--DialoGPT-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  351MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

In [14]:
def record_audio():
    fs = 16000
    duration = 10

    print("🎤 Speak now...")

    audio = sd.rec(
        int(duration * fs),
        samplerate=fs,
        channels=1
    )

    sd.wait()

    write("recorded.wav", fs, audio)

    print("✅ Recording Finished")

In [15]:
def speech_to_text():
    
    segments, info = whisper_model.transcribe(
        "recorded.wav"
    )

    text = ""

    for segment in segments:
        text += segment.text

    return text.strip()

In [16]:
def get_answer(question):

    prompt = f"Question: {question}\nAnswer:"

    response = chatbot(
        prompt,
        max_new_tokens=50,
        do_sample=True
    )

    answer = response[0]["generated_text"]

    return answer

In [17]:
def speak(text):

    speaker.say(text)
    speaker.runAndWait()

In [25]:
record_audio()

user_text = speech_to_text()

print("🧑 You:", user_text)

answer = get_answer(user_text)

print("🤖 Assistant:", answer)

speak(answer)


🎤 Speak now...
✅ Recording Finished


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧑 You: At this mission, you are near.
🤖 Assistant: Question: At this mission, you are near.
Answer: I'm still on the mission


In [26]:
user_text = speech_to_text()

if len(user_text.strip()) < 3:
    print("⚠️ No speech detected. Please speak again.")
else:
    answer = get_answer(user_text)
    print("🤖 Assistant:", answer)
    speak(answer)
    

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Assistant: Question: At this mission, you are near.
Answer: At this mission , you are as far away from the enemy as possible .
